# Week 12: Poker BC Training on Colab

**Goal:** Fine-tune Qwen 1.5B via behavior cloning on heuristic poker trajectories.

**Expected result:** Improve action accuracy from 8% (zero-shot) toward heuristic baseline (100%).

**Runtime:** ~20-30 min on T4 GPU (Colab free tier)

### Setup
1. Runtime > Change runtime type > **T4 GPU**
2. Run all cells in order

## 1. Install Dependencies & Clone Repo

In [ ]:
# Install dependencies
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes

# Clone repo
import os
if not os.path.exists('STAT-4830-RL-project'):
    !git clone https://github.com/aadithyasrinivasan/STAT-4830-RL-project.git
os.chdir('STAT-4830-RL-project')
!git pull

import sys
sys.path.insert(0, '.')
print('Setup complete.')

In [ ]:
# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
gpu_cap = torch.cuda.get_device_capability(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu_name} (compute {gpu_cap[0]}.{gpu_cap[1]}, {gpu_mem:.1f} GB)')
print(f'Precision: {"bf16" if gpu_cap[0] >= 8 else "fp16"}')

In [ ]:
# Run tests to verify everything works
!python3 -m pytest tests/ -v --tb=short 2>&1 | tail -10

## 2. Collect BC Training Data

In [ ]:
import random
from collections import Counter

from src.poker.training import collect_poker_trajectories_with_traces
from src.poker.rewards import parse_action

random.seed(42)

# Collect 500 heuristic trajectories with reasoning traces
print('Collecting 500 heuristic trajectories...')
trajectories = collect_poker_trajectories_with_traces(num_episodes=500)

correct = sum(1 for t in trajectories if t.is_correct)
parsed = sum(1 for t in trajectories if t.parsed_stats)
print(f'\nCollected: {len(trajectories)} trajectories')
print(f'Correct: {correct} ({correct/len(trajectories):.0%})')
print(f'Parsed stats: {parsed} ({parsed/len(trajectories):.0%})')

# Action distribution
action_dist = Counter(t.action_type for t in trajectories)
print(f'\nAction distribution:')
for action, count in action_dist.most_common():
    print(f'  {action}: {count} ({count/len(trajectories):.0%})')

# Context length stats
ctx_lens = [len(t.context) for t in trajectories]
code_lens = [len(t.code) for t in trajectories]
print(f'\nContext: avg {sum(ctx_lens)/len(ctx_lens):.0f} chars (range {min(ctx_lens)}-{max(ctx_lens)})')
print(f'Code: avg {sum(code_lens)/len(code_lens):.0f} chars (range {min(code_lens)}-{max(code_lens)})')

# Estimate token count
est_tokens = [(c + k + 500) / 4 for c, k in zip(ctx_lens, code_lens)]
print(f'\nEstimated total tokens per example: avg {sum(est_tokens)/len(est_tokens):.0f}, max {max(est_tokens):.0f}')

## 3. Load Model (Qwen 1.5B + LoRA)

In [ ]:
from src.training import load_model

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'

print(f'Loading {MODEL_ID} with 4-bit quantization + LoRA...')
model, tokenizer = load_model(
    model_id_or_path=MODEL_ID,
    load_in_4bit=True,
    lora_r=16,
    lora_alpha=32,
)
print(f'\nModel loaded on {model.device}')

## 4. Pre-Training Evaluation (Zero-Shot Baseline)

In [ ]:
from src.poker.agents import PokerLocalLLMAgent, PokerHeuristicAgent
from src.poker.tasks import generate_poker_task
from src.poker.evaluation import PokerEvaluationFramework

# Evaluate zero-shot performance before training
print('Evaluating zero-shot performance (25 episodes)...')

random.seed(123)
zs_agent = PokerLocalLLMAgent(
    model=model, tokenizer=tokenizer,
    name='ZeroShot-Qwen1.5B', max_steps=3, temperature=0.1,
)
heuristic = PokerHeuristicAgent()

zs_eval = PokerEvaluationFramework(
    agents=[zs_agent, heuristic],
    task_generator=generate_poker_task,
    num_episodes=25,
)
zs_eval.run_evaluation()
zs_eval.display_results()

zs_accuracy = zs_eval.results['ZeroShot-Qwen1.5B']['type_match'] / max(zs_eval.results['ZeroShot-Qwen1.5B']['total'], 1)
print(f'\nZero-shot accuracy: {zs_accuracy:.1%}')

## 5. Run BC Training

In [ ]:
from src.poker.training import PokerBCTrainer
import time

BC_OUTPUT = './checkpoints/poker_bc'

bc_trainer = PokerBCTrainer(
    model=model,
    tokenizer=tokenizer,
    output_dir=BC_OUTPUT,
    num_epochs=3,
    batch_size=2,       # small batch for T4 memory
    learning_rate=2e-4,
    max_length=4096,    # enough for full poker context + code
)

print(f'Starting BC training: {len(trajectories)} trajectories, 3 epochs')
print(f'Estimated time: ~15-25 minutes on T4\n')

start_time = time.time()
bc_result = bc_trainer.train(trajectories)
elapsed = time.time() - start_time

print(f'\n{"="*50}')
print(f'BC TRAINING COMPLETE ({elapsed/60:.1f} min)')
print(f'  Final loss: {bc_result["train_loss"]:.4f}')
print(f'  Examples: {bc_result["num_examples"]}')
print(f'  Saved to: {bc_result["output_dir"]}')
print(f'{"="*50}')

## 6. Post-Training Evaluation (BC Model)

In [ ]:
# Evaluate the BC-trained model
print('Evaluating BC model (50 episodes)...')

random.seed(456)
bc_agent = PokerLocalLLMAgent(
    model=model, tokenizer=tokenizer,
    name='BC-Qwen1.5B', max_steps=3, temperature=0.1,
)
heuristic = PokerHeuristicAgent()

bc_eval = PokerEvaluationFramework(
    agents=[bc_agent, heuristic],
    task_generator=generate_poker_task,
    num_episodes=50,
)
bc_eval.run_evaluation()
bc_eval.display_results()

print('\n--- Confusion Matrix (BC Model) ---')
bc_eval.display_confusion_matrix('BC-Qwen1.5B')

bc_accuracy = bc_eval.results['BC-Qwen1.5B']['type_match'] / max(bc_eval.results['BC-Qwen1.5B']['total'], 1)
print(f'\n{"="*50}')
print(f'IMPROVEMENT: {zs_accuracy:.1%} (zero-shot) -> {bc_accuracy:.1%} (BC)')
print(f'{"="*50}')

## 7. Per-Street Evaluation

In [ ]:
from src.poker.tasks import generate_preflop_task, generate_postflop_task

print('=== PREFLOP EVALUATION ===')
random.seed(789)
pre_eval = PokerEvaluationFramework(
    agents=[bc_agent, heuristic],
    task_generator=generate_preflop_task,
    num_episodes=30,
)
pre_eval.run_evaluation()
pre_eval.display_results()

print('\n=== POSTFLOP EVALUATION ===')
random.seed(101)
post_eval = PokerEvaluationFramework(
    agents=[bc_agent, heuristic],
    task_generator=generate_postflop_task,
    num_episodes=30,
)
post_eval.run_evaluation()
post_eval.display_results()

## 8. Results Summary & Visualization

In [ ]:
import json
import os

# Compile all results
results_summary = {
    'zero_shot': {
        'accuracy': zs_accuracy,
        'episodes': zs_eval.results.get('ZeroShot-Qwen1.5B', {}).get('total', 0),
        'by_action': {k: v for k, v in zs_eval.results.get('ZeroShot-Qwen1.5B', {}).get('by_action', {}).items()},
        'by_street': {k: v for k, v in zs_eval.results.get('ZeroShot-Qwen1.5B', {}).get('by_street', {}).items()},
    },
    'bc_model': {
        'accuracy': bc_accuracy,
        'episodes': bc_eval.results.get('BC-Qwen1.5B', {}).get('total', 0),
        'train_loss': bc_result['train_loss'],
        'train_examples': bc_result['num_examples'],
        'train_epochs': bc_result['epochs'],
        'by_action': {k: v for k, v in bc_eval.results.get('BC-Qwen1.5B', {}).get('by_action', {}).items()},
        'by_street': {k: v for k, v in bc_eval.results.get('BC-Qwen1.5B', {}).get('by_street', {}).items()},
    },
    'heuristic': {'accuracy': 1.0},
    'model': MODEL_ID,
    'training_time_min': elapsed / 60,
}

os.makedirs('experiments/results', exist_ok=True)
with open('experiments/results/week12_bc_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)
print('Results saved to experiments/results/week12_bc_results.json')

# Print final summary table
print(f'\n{"="*60}')
print(f'FINAL RESULTS SUMMARY')
print(f'{"="*60}')
print(f'{"Agent":<25} {"Accuracy":>10} {"Episodes":>10}')
print(f'{"-"*45}')
print(f'{"Zero-Shot (Qwen-7B)":<25} {"8.0%":>10} {"25":>10}')
print(f'{"Zero-Shot (Qwen-1.5B)":<25} {zs_accuracy:>10.1%} {"25":>10}')
print(f'{"BC (Qwen-1.5B+LoRA)":<25} {bc_accuracy:>10.1%} {"50":>10}')
print(f'{"Heuristic (ceiling)":<25} {"100.0%":>10} {"any":>10}')
print(f'{"="*60}')

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Plot 1: Improvement arc
    agents = ['Zero-Shot\n(Qwen-7B)', 'Zero-Shot\n(Qwen-1.5B)', 'BC Model\n(Qwen-1.5B)', 'Heuristic\n(ceiling)']
    accs = [8.0, zs_accuracy * 100, bc_accuracy * 100, 100.0]
    colors = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
    bars = axes[0].bar(range(len(agents)), accs, color=colors)
    axes[0].set_xticks(range(len(agents)))
    axes[0].set_xticklabels(agents, fontsize=9)
    axes[0].set_ylabel('Action Accuracy (%)')
    axes[0].set_title('Improvement Arc')
    axes[0].set_ylim(0, 115)
    for bar, acc in zip(bars, accs):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                     f'{acc:.1f}%', ha='center', fontsize=10, fontweight='bold')

    # Plot 2: BC model per-action accuracy
    bc_by_action = bc_eval.results.get('BC-Qwen1.5B', {}).get('by_action', {})
    if bc_by_action:
        action_names = sorted(bc_by_action.keys())
        action_accs = [bc_by_action[a]['correct'] / max(bc_by_action[a]['total'], 1) * 100 for a in action_names]
        action_totals = [bc_by_action[a]['total'] for a in action_names]
        bars2 = axes[1].bar(action_names, action_accs, color='#3498db')
        axes[1].set_ylabel('Accuracy (%)')
        axes[1].set_title('BC Model: Per-Action Accuracy')
        axes[1].set_ylim(0, 115)
        for bar, acc, n in zip(bars2, action_accs, action_totals):
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                         f'{acc:.0f}%\n(n={n})', ha='center', fontsize=9)

    # Plot 3: Confusion matrix heatmap
    matrix = bc_eval.get_confusion_matrix('BC-Qwen1.5B')
    action_order = ['fold', 'check', 'call', 'raise']
    mat = np.array([[matrix[t][p] for p in action_order] for t in action_order])
    im = axes[2].imshow(mat, cmap='Blues', aspect='auto')
    axes[2].set_xticks(range(4))
    axes[2].set_xticklabels(action_order)
    axes[2].set_yticks(range(4))
    axes[2].set_yticklabels(action_order)
    axes[2].set_xlabel('Predicted')
    axes[2].set_ylabel('Correct')
    axes[2].set_title('BC Model: Confusion Matrix')
    for i in range(4):
        for j in range(4):
            axes[2].text(j, i, str(mat[i, j]), ha='center', va='center',
                         color='white' if mat[i, j] > mat.max()/2 else 'black')

    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/week12_bc_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved to figures/week12_bc_results.png')
except ImportError:
    print('matplotlib not available — skipping plots')

## 9. Example BC Model Outputs

Show what the trained model actually generates.

In [ ]:
# Show a few example predictions from the BC model
random.seed(999)

for i in range(5):
    context, question, answer = generate_poker_task()
    predicted, transcript = bc_agent.run_episode(context, question, answer)

    # Detect street
    street = 'preflop'
    for s in ['River)', 'Turn)', 'Flop)']:
        if s in context:
            street = s.replace(')', '').lower()
            break

    match = '  ' if parse_action(predicted)[0] == parse_action(answer)[0] else 'X '
    print(f'{match}Example {i+1} [{street:7s}]: correct={answer:<12s} predicted={predicted:<12s}')

    # Show last code block from transcript
    for entry in transcript:
        if 'code' in entry:
            code_preview = entry['code'][:200].replace('\n', '\n    ')
            print(f'    Code: {code_preview}...')
            if entry.get('exec_result', {}).get('stdout'):
                print(f'    Output: {entry["exec_result"]["stdout"].strip()[:150]}')
    print()

## 10. Save Checkpoint to Drive (Optional)

Save the trained model to Google Drive for later use.

In [ ]:
# Uncomment to save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r ./checkpoints/poker_bc /content/drive/MyDrive/poker_bc_checkpoint
# print('Checkpoint saved to Google Drive')

print('To save: uncomment the lines above and run this cell.')
print(f'\nCheckpoint location: {BC_OUTPUT}')
print(f'Files:')
!ls -la {BC_OUTPUT}/ 2>/dev/null || echo 'Checkpoint not yet created (run training first)'

## 11. Next Steps

After BC training:
1. **If BC accuracy > 30%**: Proceed to REINFORCE (Phase 2) from BC checkpoint
2. **If BC accuracy < 30%**: Increase training data (1000+ episodes), try 5 epochs, lower learning rate
3. **REINFORCE**: `python scripts/poker_train.py --phase rl --model ./checkpoints/poker_bc`
4. **Final evaluation**: Compare zero-shot vs BC vs RL vs heuristic across all streets